# 70 — Train BGE-M3 bi-encoder (Stage A) — end-to-end

Single-path notebook. Fine-tunes a pretrained dense retriever on a fraction
of HF train,
trains on a user-disjoint train/val split, pushes the merged model
to HF Hub, re-embeds the ~47K-track catalog, and reports standalone
dev nDCG@20.

## Quick start

Set `SKIP_MINING = True` (default) in cell 1 to reuse an existing
`TRIPLES_JSONL` and run cells 2-7 end-to-end. Set `SKIP_MINING = False`
to force a fresh HN mine.


1. Set the parameters in **cell 1 (CONFIG)** — `MAX_INPUT_ROWS`, `EPOCHS`,
   `PER_DEVICE_BATCH_SIZE`, etc. — to control the run size.
2. Run cells 2-7 sequentially. Open cell 5 (TensorBoard) in parallel with cell 4 (train).

## What we're verifying

1. **train/val nDCG alignment** — `train/ndcg_inbatch` and `val/ndcg`
   curves should track each other (gap < ~0.05 sustained). Gap > 0.05 = leak.
2. **honest retrieval lift** — `val/full_catalog_ndcg_at_20` (vs the real ~47K
   catalog) should climb above ~0.05-0.08 by end of epoch 1.
3. **HF train/test disjointness** — preflight in cell 7 refuses to score
   if HF train and test splits share user_ids or session_ids.

## Warm-start across sessions

Per-epoch LoRA checkpoints save to `TRAIN_OUTPUT_DIR/checkpoint_epoch_{1,2,3}/` on
Drive — they survive Colab session restarts. To resume training in a future session,
set `RESUME_FROM` in cell 1 to one of those paths and run cells 1–7.

## Recent patches (2026-05-22)

- **`BGE_MODEL` backbone knob** in CONFIG. Current default `BAAI/bge-base-en-v1.5`
  (110M, English-only) — 5× smaller than BGE-M3, allows much larger batches.
  Set to `BAAI/bge-m3` to revert.


After BlindA nDCG regressed (0.06 baseline → 0.05 fine-tuned), a systematic-
debugging audit found TWO logical bugs. Both are fixed in this commit:

- **Mining filter selection bias** — `percpos@0.95` skipped 60% of queries
  (the hard ones), biasing the training distribution toward easy queries.
  Fix: new CONFIG knob `MINING_STRATEGY = 'percpos' | 'simans'`. Switch to
  SimANS to eliminate the filter and use 100% of queries.
- **Catalog vs [HISTORY] format mismatch** — pos/neg used `format_track_text`
  (5 fields, pipe-separated, original case) while [HISTORY] music-turn
  references used `id_to_metadata` (4 fields, comma, lowercased). The
  encoder had to learn two representations of every track. Fix: builder
  and cell 6 now both use `id_to_metadata` format → everything aligned.

**To take effect**, set `MINING_STRATEGY = 'simans'` in CONFIG, then re-mine
(cell 3 with SKIP_MINING=False) + re-embed (cell 6) + retrain (cell 4).

## §6.5 amendment knobs active in this run

- `--split-key user_id` — user-disjoint train/val (every session of a given user lives in one partition).
- `UserDisjointBatchSampler` — distinct users per batch; without-replacement; heap-greedy load-balanced.
- In-batch InfoNCE with false-positive collision masking (same-pos-tid + cross-pos-into-neg-slot; RocketQAv2 / BGE-M3 §3.3).
- Per-row `K_data = N_NEGATIVES` mined negs; rows below threshold are dropped (no upsampling).


In [ ]:
# 1) CONFIG — all run parameters in one place. Edit then run cells 2-7.

# CRITICAL: set GPU memory env vars BEFORE any imports that pull JAX/TF.
# Colab's `datasets` + `transformers` transitively import JAX, which
# preallocates 90% of GPU memory by default (~85 GB out of 95 GB on
# Blackwell). That blocks the training subprocess in cell 4 from getting
# enough VRAM → OOM. These vars force growth-only mode. MUST be set in the
# kernel BEFORE the first datasets/transformers/jax import — otherwise no
# effect (you'd need to restart the runtime).
import os
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
os.environ.setdefault('TF_FORCE_GPU_ALLOW_GROWTH', 'true')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

# --- Branch + identity --------------------------------------------------
BRANCH       = 'fresh-model'
HUB_USER     = 'OrRim123'
RUN_NAME     = 'bge-base-en-music-v1-mm'  # multi-modal fork; preserves the text-only -v1 artifacts
                                  # IMPORTANT: also change RUN_NAME when changing BGE_MODEL below,
                                  # so the Hub repo name reflects the actual model architecture.

BGE_MODEL    = 'BAAI/bge-base-en-v1.5'  # Pre-trained encoder backbone. Tested options:
                                  #   'BAAI/bge-m3'             — 567M params, 1024-d, multilingual (default)
                                  #   'BAAI/bge-base-en-v1.5'   — 110M params,  768-d, English-only (5× smaller;
                                  #     allows MUCH larger batch sizes — try MINING_BATCH_SIZE=256,
                                  #     PER_DEVICE_BATCH_SIZE=64, GRAD_ACCUM_STEPS=1 for ~3× faster training)
                                  #   'BAAI/bge-large-en-v1.5'  — 335M params, 1024-d, English-only

HUB_REPO_OVERRIDE = ''  # use auto-derived '{HUB_USER}/recsys2026-{RUN_NAME}'; protects v1 repo from being overwritten
                                  # (was '' = auto-derive). Set to '' to revert. Use cases:
                                  #   - Reuse an existing repo you have write access to (e.g. v1) instead
                                  #     of creating a new one. Train script will overwrite that repo's
                                  #     contents with the newly-trained model.
                                  #   - Avoid the 'repo doesn't exist yet' 404 when running cell 6/7
                                  #     against an existing Hub repo before re-training.
                                  # Format: '<HUB_USER>/recsys2026-<some-name>' (script appends '-merged')
                                  # Example: 'OrRim123/recsys2026-bge-m3-music-v1' → pushes to
                                  #          'OrRim123/recsys2026-bge-m3-music-v1-merged' (existing repo)
                                  # Empty = use auto-derived 'OrRim123/recsys2026-{RUN_NAME}'.

# --- Data scope ---------------------------------------------------------
SKIP_MINING    = False          # full run: re-mine. Set True to reuse an existing TRIPLES_JSONL after the first run.
MAX_INPUT_ROWS = 0              # full ~121K run (~6-8 hr on Blackwell). Set 20000 for a ~1 hr scaled-up smoke.
DEV_EVAL_ROWS  = 8000           # # of HF test rows scored in cell 7 (use full HF test split)

# --- HN mining ----------------------------------------------------------
MINING_STRATEGY   = 'simans'    # 'percpos' (filter; ~60% skip rate at threshold=0.95) or
                                # 'simans'  (Gaussian-weighted, no skip; uses 100% of queries).
                                # SimANS recommended when val/dev/BlindA show train-data
                                # selection bias (high val nDCG but low dev/BlindA nDCG).
PERCPOS_THRESHOLD = 0.95        # candidates KEPT if score < threshold * pos_score (percpos only)
POOL_SIZE         = 1000        # top-K candidates considered per query
N_NEGATIVES       = 15          # negs per row (rows below this are DROPPED at train time)
MINING_BATCH_SIZE = 256         # 4× larger than BGE-M3 default (bge-base is 5× smaller)
SIMANS_A          = 0.1         # SimANS Gaussian peak offset (s_pos - a); used when MINING_STRATEGY='simans'
SIMANS_B          = 0.05        # SimANS Gaussian spread; smaller = narrower peak around target

# --- Training -----------------------------------------------------------
EPOCHS                     = 1        # 3 for production; 1 for quick-iter sanity (start here)
SEED                       = 42       # master seed (data split, samplers, LoRA init)
PER_DEVICE_BATCH_SIZE      = 32       # bge-base + LoRA-on-FFN (intermediate.dense, 768→3072) hits ~2.4GB/layer
                                      # FP32 cast at bs=64. bs=32 fits comfortably. Bump only after confirming headroom.
GRAD_ACCUM_STEPS           = 4        # was 2; effective batch = 32 * 4 = 128 (smoother gradient)
LR                         = 2e-5     # was 5e-6; LoRA can use 4× higher LR than full FT (PEFT literature)
LR_SCHEDULE                = 'cosine' # was 'linear'; cosine matches BGE-M3/E5/GTE recipes (smoother LR decay)
TEMPERATURE                = 0.02     # was 0.05; BGE-M3 paper recipe (sharper softmax = stronger hard-neg gradient)
LORA_RANK                  = 128      # was 64; doubled capacity (still safe: 21M trainable / 115K rows = 183 params/row)
LORA_ALPHA                 = 256      # convention: 2 × LoRA_RANK
QUERY_MAX_LEN              = 384
PASSAGE_MAX_LEN            = 192
SPLIT_KEY                  = 'user_id'   # 'user_id' | 'session_id' | 'row'
VAL_FRACTION               = 0.10
LOGGING_STEPS              = 25       # log every N opt-steps; 5 for quick-iter
VAL_EVERY_N_STEPS          = 100      # val pass every N opt-steps; 10 for quick-iter
VAL_FULL_CATALOG_EVERY_N   = 200      # full-catalog (~50 sec each) every N opt-steps; 50 for quick-iter
CHECKPOINT_EVERY_N_EPOCHS  = 1        # save adapter per epoch (~30 MB each) for warm-start/revert
RESUME_FROM                = ''       # '' = train from scratch; otherwise absolute path to a checkpoint_epoch_N/
                                      # directory under TRAIN_OUTPUT_DIR. Optimizer/scheduler restart fresh
                                      # (script default); --epochs counts ADDITIONAL epochs from the checkpoint.
GRADIENT_CHECKPOINTING     = True
TENSORBOARD_PORT           = 6006

# --- Multi-modal upgrade (fresh-model branch) ---------------------------
# Toggle the end-to-end multi-modal pipeline (text + CLAP audio + CF + tags
# + release-year + user CF). Default is False → reverts to the current
# text-only baseline so existing comparisons keep running. Flip to True
# AFTER cells 1b (artifacts) and 1c (teacher scores) have run successfully.
USE_MULTIMODAL          = True      # master toggle for the multi-modal path
MULTIMODAL_ARTIFACTS    = '/content/drive/MyDrive/recsys2026_retrieval_v2_cache/multimodal'
                                    # shared cache dir for tag_vocab / track_clap /
                                    # track_cf / user_cf / user_cf_mean / release_year
USE_DISTILLATION        = True      # MarginMSE loss term from teacher_scores (Phase 0c)
DISTILL_WEIGHT          = 0.3       # 0.7 * InfoNCE + 0.3 * MarginMSE
USE_MULTIPOSITIVE       = True      # multi-positive InfoNCE (teacher promotes alts)
MULTIPOSITIVE_THRESHOLD = 0.85      # candidate -> positive iff score >= 0.85 * gold_score
MODALITY_DROPOUT        = 0.1       # per-batch chance to zero one modality (regularization)
TEACHER_MODEL           = 'BAAI/bge-reranker-v2-m3'
TEACHER_SCORES_PATH     = f'{MULTIMODAL_ARTIFACTS}/teacher_scores_{RUN_NAME.replace("-","_")}.parquet'

# --- Derived paths (do not edit usually) --------------------------------
HUB_REPO         = HUB_REPO_OVERRIDE if HUB_REPO_OVERRIDE else f'{HUB_USER}/recsys2026-{RUN_NAME}'
HUB_REPO_MERGED  = f'{HUB_REPO}-merged'
EMBED_LABEL      = f'{RUN_NAME}-merged'
TRIPLES_JSONL    = f'experiments/cache/retrieval_v2/triples_{RUN_NAME.replace("-","_")}.jsonl'
# Training artifacts (LoRA adapter + checkpoint_epoch_*/ + runs/) live on
# Drive so per-epoch checkpoints SURVIVE Colab session restarts → enables
# cross-session warm-start via RESUME_FROM above.
TRAIN_OUTPUT_DIR = f'/content/drive/MyDrive/recsys2026_retrieval_v2_cache/training/{RUN_NAME.replace("-","_")}'
DRIVE_RESULTS    = f'/content/drive/MyDrive/recsys2026_retrieval_v2_cache/results/{RUN_NAME.replace("-","_")}'
DRIVE_LOG_HN     = f'/content/drive/MyDrive/recsys2026_retrieval_v2_cache/{RUN_NAME.replace("-","_")}_hn_log.txt'
DRIVE_LOG_TRAIN  = f'/content/drive/MyDrive/recsys2026_retrieval_v2_cache/{RUN_NAME.replace("-","_")}_train_log.txt'
CACHE_ROOT       = '/content/drive/MyDrive/recsys2026_retrieval_v2_cache/dense_local'
SAFE_MODEL       = HUB_REPO_MERGED.replace('/', '_')
CATALOG_OUT_DIR  = f'{CACHE_ROOT}/{SAFE_MODEL}/{EMBED_LABEL}'

print('Config loaded:')
for k in ('BRANCH','BGE_MODEL','HUB_REPO_OVERRIDE','HUB_REPO','HUB_REPO_MERGED','TRIPLES_JSONL','TRAIN_OUTPUT_DIR',
         'SKIP_MINING','RESUME_FROM','SEED','MAX_INPUT_ROWS','EPOCHS','PER_DEVICE_BATCH_SIZE','GRAD_ACCUM_STEPS',
         'CHECKPOINT_EVERY_N_EPOCHS','N_NEGATIVES','MINING_STRATEGY','PERCPOS_THRESHOLD','SPLIT_KEY','VAL_FRACTION','LR','LR_SCHEDULE','TEMPERATURE','LORA_RANK',
         'USE_MULTIMODAL','MULTIMODAL_ARTIFACTS','USE_DISTILLATION','DISTILL_WEIGHT','USE_MULTIPOSITIVE','MULTIPOSITIVE_THRESHOLD','MODALITY_DROPOUT','TEACHER_MODEL','TEACHER_SCORES_PATH'):
    print(f'  {k} = {globals()[k]!r}')



In [ ]:
# 2) Setup — clone branch + HF auth + Drive mount + deps.
import os
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive', force_remount=True)   # tolerate Colab's leftover /content/drive state on session reconnects

!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

DRIVE_BASE = '/content/drive/MyDrive'
LOCAL_BASE = '/content/recsys2026/experiments/cache'
os.makedirs(LOCAL_BASE, exist_ok=True)
for name, drive_subdir in [
    ('sid', 'recsys2026_sid_cache'),
    ('retrieval_v2', 'recsys2026_retrieval_v2_cache'),
    ('dense_local', 'recsys2026_retrieval_v2_cache/dense_local'),  # multi-modal: closes path mismatch where cell 6 wrote to Drive but cell 7 read from local
]:
    src = f'{DRIVE_BASE}/{drive_subdir}'
    dst = f'{LOCAL_BASE}/{name}'
    os.makedirs(src, exist_ok=True)
    if os.path.islink(dst): os.unlink(dst)
    elif os.path.exists(dst):
        import shutil; shutil.rmtree(dst)
    os.symlink(src, dst)

!pip install -q --upgrade \
    'peft>=0.10' 'transformers>=4.40' 'accelerate>=0.30' \
    'sentence-transformers>=3.0' 'FlagEmbedding>=1.3' 'bm25s' 'torchao>=0.16' \
    'datasets' 'pandas<3.0' 'tqdm' 'omegaconf' 'pyyaml' 'tensorboard'


In [ ]:
# 2b) Precompute shared multi-modal artifacts (tag_vocab, CLAP, CF, user CF).
# One-time builder for the artifacts that Stage A (multi-modal bi-encoder) and
# Stage B (multi-modal cross-encoder) both consume at train + inference time.
# Idempotent — each artifact is built only if missing. Safe to re-run.
# Skip if you're sticking with the text-only baseline (USE_MULTIMODAL=False).
if USE_MULTIMODAL:
    !cd /content/recsys2026 && python -u scripts/precompute_multimodal_artifacts.py \
        --cache-dir {MULTIMODAL_ARTIFACTS}
    !ls -la {MULTIMODAL_ARTIFACTS}
else:
    print('[1b] USE_MULTIMODAL=False — skipping artifact precompute')


In [ ]:
# 3) HN mine — produces TRIPLES_JSONL from HF train.
# Honors SKIP_MINING: if the JSONL already exists on Drive and the flag
# is True, this cell skips the ~25-85 min mine and just confirms the
# file's there. Set SKIP_MINING = False in cell 1 to force a re-mine.
#
# PATCH 4 (2026-05-22): the builder now writes pos/neg in id_to_metadata
# format (matching cell 6's catalog re-embed + the [HISTORY] expansion).
# Old triples_*.jsonl files mined before this patch have pos/neg in the
# legacy 5-field format — set SKIP_MINING=False to re-mine, OR keep
# SKIP_MINING=True if you want to compare against the old format.
import os
RESULTS_DIR = '/content/drive/MyDrive/recsys2026_retrieval_v2_cache/results'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DRIVE_RESULTS, exist_ok=True)

_existing_triples = os.path.exists(TRIPLES_JSONL)
if SKIP_MINING and _existing_triples:
    print(f'[mine] SKIP_MINING=True and {TRIPLES_JSONL} exists — skipping mine.')
    !wc -l {TRIPLES_JSONL}
    !head -1 {TRIPLES_JSONL} | python3 -c "import json,sys; r=json.loads(sys.stdin.read()); print('HAS session_id:', 'session_id' in r); print('HAS user_id:', 'user_id' in r); print('HAS neg_tids:', 'neg_tids' in r); print('pos[0] sample:', r['pos'][0][:200]); print('keys:', sorted(r.keys()))"
else:
    if SKIP_MINING and not _existing_triples:
        print(f'[mine] SKIP_MINING=True but {TRIPLES_JSONL} not found — mining anyway.')
    !rm -f {TRIPLES_JSONL}
    # Re-mining invalidates the teacher_scores parquet (its key is
    # (pos_tid, sorted(neg_tids)) and the new triples will have different
    # neg samples). Clear it so cell 3d re-precomputes from scratch
    # rather than resuming on stale rows. Cell 3d is idempotent on a
    # clean partial.
    if USE_MULTIMODAL:
        !rm -f {TEACHER_SCORES_PATH} {TEACHER_SCORES_PATH}.partial
        print(f'[mine] cleared stale teacher scores at {TEACHER_SCORES_PATH} (re-mine will produce different neg_tids)')
    _max_rows_flag = f'--max-rows {MAX_INPUT_ROWS}' if MAX_INPUT_ROWS > 0 else ''
    _strategy_flags = f'--mining-strategy {MINING_STRATEGY}'
    if MINING_STRATEGY == 'simans':
        _strategy_flags += f' --simans-a {SIMANS_A} --simans-b {SIMANS_B}'
    # Multi-modal flag (Phase 1, fresh-model): when USE_MULTIMODAL=True,
    # the builder emits tag_ids_{pos,neg} + release_year_{pos,neg} fields
    # alongside the legacy text triples (no behavioral change otherwise).
    _mm_flag = f'--multimodal-artifacts {MULTIMODAL_ARTIFACTS}' if USE_MULTIMODAL else ''
    print(f'[mine] strategy={MINING_STRATEGY} (percpos→filtered+skip; simans→no-skip, all queries kept); USE_MULTIMODAL={USE_MULTIMODAL}')
    !cd /content/recsys2026 && python -u scripts/build_bi_encoder_training_data.py \
        --train-conv-hf talkpl-ai/TalkPlayData-Challenge-Dataset \
        --bge-m3-model {BGE_MODEL} \
        --output {TRIPLES_JSONL} \
        --query-mode bge_m3_structured \
        {_strategy_flags} \
        --percpos-threshold {PERCPOS_THRESHOLD} --pool-size {POOL_SIZE} \
        --k-negs {N_NEGATIVES} --batch-size {MINING_BATCH_SIZE} \
        {_max_rows_flag} \
        {_mm_flag} \
        2>&1 | tee {DRIVE_LOG_HN}
    !wc -l {TRIPLES_JSONL}
    !head -1 {TRIPLES_JSONL} | python3 -c "import json,sys; r=json.loads(sys.stdin.read()); print('HAS session_id:', 'session_id' in r); print('HAS user_id:', 'user_id' in r); print('HAS neg_tids:', 'neg_tids' in r); print('pos[0] sample:', r['pos'][0][:200]); print('keys:', sorted(r.keys()))"


In [ ]:
# 3b) (One-shot) Expand music-turn IDs in [HISTORY]: blocks to match
# production inference (id_to_metadata format). Closes the train/eval
# parity gap on the [HISTORY] music turns.
#
# This is idempotent and FAST (~1-2 min CPU): only the `query` field is
# rewritten; all other fields (pos/neg/pos_tid/neg_tids/user_id/session_id)
# are preserved byte-for-byte.
#
# After the builder fix in commit (see history-corpus-types arg), fresh
# mines from cell 3 already emit the corrected format → this cell becomes
# a no-op (rewrites 0 rows). Kept for safety + to salvage any pre-fix
# JSONL still around on Drive.
TRIPLES_FIXED = TRIPLES_JSONL.replace('.jsonl', '_history_fixed.jsonl')
!cd /content/recsys2026 && python -u scripts/expand_history_in_triples.py \
    --input {TRIPLES_JSONL} \
    --output {TRIPLES_FIXED} \
    --track-meta-hf talkpl-ai/TalkPlayData-Challenge-Track-Metadata \
    --corpus-types track_name,artist_name,album_name
# Swap in the fixed file so cell 4 (train) picks it up automatically.
!mv {TRIPLES_FIXED} {TRIPLES_JSONL}
# Confirm one row's [HISTORY] now contains `A: track_id: <id>, track_name:...`.
!python3 -c "import json; r=json.loads(open('{TRIPLES_JSONL}').readline()); print(r['query'][:600])"


In [ ]:
# 3c) DIAGNOSTIC — s_pos distribution: TRAIN-mined vs DEV.
#
# Why: percpos mining filters out queries where the gold doesn't stand
# out from a 1000-candidate pool. The 60% skip rate at threshold=0.95
# creates a curated training subset biased toward HIGH s_pos queries.
# Dev (HF test) and BlindA include the FULL s_pos distribution.
# This cell quantifies that bias by computing zero-shot s_pos via BGE_MODEL for
# a sample of TRAIN-mined queries vs DEV queries.
#
# Read with: 'how much smaller is the train-mined s_pos distribution's
# tail than dev's tail?' Big gap → mining filter selection bias confirmed.
# Skippable for production runs — uncomment the early `raise` to disable.
# raise SystemExit('diagnostic disabled')
import json, numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import sys
sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
sys.path.insert(0, '/content/recsys2026/scripts')
from mcrs.retrieval_modules.bge_m3_format import format_query_text
from build_bi_encoder_training_data import _iter_conversation_turns, _format_history_music_turn

DIAG_SAMPLE = 200  # queries per side

# Load catalog + encode with zero-shot BGE-M3 (same encoder the mining used).
tm = load_dataset('talkpl-ai/TalkPlayData-Challenge-Track-Metadata', split='all_tracks')
metadata_dict = {row['track_id']: dict(row) for row in tm}
corpus_types = ['track_name', 'artist_name', 'album_name']
track_ids = [row['track_id'] for row in tm]
track_texts = [_format_history_music_turn(tid, metadata_dict, corpus_types) for tid in track_ids]
tid_to_idx = {tid: i for i, tid in enumerate(track_ids)}

# Uses BGE_MODEL (from CONFIG) so the diagnostic matches whichever encoder mining used.
# Works for any sentence-transformers-compatible model (BGE-M3, bge-base-en, etc.).
print(f'[diag] loading zero-shot encoder: {BGE_MODEL}')
model = SentenceTransformer(BGE_MODEL, device='cuda')
model.max_seq_length = 512
model = model.half()  # FP16; matches build script's mining encoder for apples-to-apples s_pos
print(f'[diag] encoding catalog ({len(track_ids):,} tracks)')
track_embs = np.asarray(model.encode(track_texts, batch_size=64,
                                     normalize_embeddings=True, convert_to_numpy=True,
                                     show_progress_bar=False), dtype=np.float32)
track_embs /= np.clip(np.linalg.norm(track_embs, axis=1, keepdims=True), 1e-9, None)

def _sample_s_pos(rows, label):
    rows = rows[:DIAG_SAMPLE]
    queries = [format_query_text(r.get('chat_history') or [], r.get('current_user_query',''),
                                  r.get('user_profile_raw'), r.get('conversation_goal'),
                                  mode='bge_m3_structured') for r in rows]
    q_embs = np.asarray(model.encode(queries, batch_size=32,
                                     normalize_embeddings=True, convert_to_numpy=True,
                                     show_progress_bar=False), dtype=np.float32)
    q_embs /= np.clip(np.linalg.norm(q_embs, axis=1, keepdims=True), 1e-9, None)
    s_pos_vals = []
    for q_emb, row in zip(q_embs, rows):
        gold = row.get('track_id')
        if gold not in tid_to_idx: continue
        s_pos = float(q_emb @ track_embs[tid_to_idx[gold]])
        s_pos_vals.append(s_pos)
    if not s_pos_vals:
        print(f'[diag] {label}: no valid s_pos values'); return None
    arr = np.asarray(s_pos_vals)
    print(f'[diag] {label}: n={len(arr)} mean={arr.mean():.3f} std={arr.std():.3f} '
          f'p10={np.percentile(arr,10):.3f} p50={np.percentile(arr,50):.3f} p90={np.percentile(arr,90):.3f}')
    return arr

print(f'\n=== TRAIN-mined sample (from {TRIPLES_JSONL}) ===')
# Match by (session_id, pos_tid) — track_id alone is ambiguous when the same
# gold track appears as the gold in multiple sessions (some of which the
# mining filter kept, others skipped). Using the tuple gives us EXACTLY
# the rows that survived mining.
train_mined_keys = set()
with open(TRIPLES_JSONL) as f:
    for line in f:
        r = json.loads(line)
        sid = r.get('session_id')
        if sid is not None:
            train_mined_keys.add((sid, r['pos_tid']))
train_ds_hf = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='train')
train_rows_all = _iter_conversation_turns(train_ds_hf, metadata_dict=metadata_dict, corpus_types=corpus_types)
train_rows_mined = [
    r for r in train_rows_all
    if (r.get('session_id'), r['track_id']) in train_mined_keys
][:DIAG_SAMPLE * 3]
import random
random.Random(42).shuffle(train_rows_mined)
_sample_s_pos(train_rows_mined, 'TRAIN-mined')

print(f'\n=== DEV sample (HF test split, unfiltered) ===')
dev_ds_hf = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='test')
dev_rows = _iter_conversation_turns(dev_ds_hf, metadata_dict=metadata_dict, corpus_types=corpus_types)
_sample_s_pos(dev_rows, 'DEV (HF test)')

print('\n[diag] Interpretation:')
print('  - If TRAIN-mined mean s_pos is MUCH higher than DEV mean → mining filter')
print('    selection bias is real (the model only trains on "easy" queries).')
print('  - Switch MINING_STRATEGY = "simans" in cell 1 to eliminate the filter.')
print('  - SimANS keeps all queries; expected: train and dev distributions match.')
# Aggressive cleanup so cell 4's subprocess gets back the GPU memory.
# del every large tensor; gc; empty CUDA cache.
del model
try: del track_embs
except NameError: pass
try: del track_ids
except NameError: pass
try: del metadata_dict
except NameError: pass
import gc, torch; gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()


In [ ]:
# 3d) Precompute teacher reranker scores for every (query, doc) pair in TRIPLES_JSONL.
# Used by Stage A for:
#   - Multi-positive label promotion (candidate -> positive iff score >= MULTIPOSITIVE_THRESHOLD * gold_score)
#   - MarginMSE distillation loss (student margin matches teacher margin)
# Wall-clock: ~2-3 hr on Blackwell bf16 for ~115K rows × 16 pairs.
# Resumable: writes to {TEACHER_SCORES_PATH}.partial line-by-line, finalizes on completion.
# Skip if USE_MULTIMODAL=False or USE_DISTILLATION+USE_MULTIPOSITIVE both False.
if USE_MULTIMODAL and (USE_DISTILLATION or USE_MULTIPOSITIVE):
    import os as _os_t
    _os_t.makedirs(MULTIMODAL_ARTIFACTS, exist_ok=True)
    !cd /content/recsys2026 && python -u scripts/precompute_reranker_scores.py \
        --triples {TRIPLES_JSONL} \
        --output {TEACHER_SCORES_PATH} \
        --model {TEACHER_MODEL} \
        --rows-per-batch 4
    !ls -la {TEACHER_SCORES_PATH}
else:
    print('[1c] USE_MULTIMODAL=False or both distill+multipos disabled — skipping teacher score precompute')


In [ ]:
# 4) Train + merge LoRA into base + push to Hub.
# Hub target: <HUB_REPO_MERGED> (gets overwritten each run).
# Open cell 5 (TensorBoard) in PARALLEL after this cell starts.
# Warm-start support: when RESUME_FROM is non-empty AND the path exists,
# pass --resume-from so training continues from that LoRA adapter.
# Optimizer/scheduler restart fresh (script-side intentional choice).
import os as _os_warmstart
_resume_flag = ''
if RESUME_FROM:
    if _os_warmstart.path.isdir(RESUME_FROM):
        _resume_flag = f'--resume-from {RESUME_FROM}'
        print(f'[train] WARM-START from {RESUME_FROM}')
    else:
        raise FileNotFoundError(
            f'RESUME_FROM is set to {RESUME_FROM!r} but the path does not exist. '
            f'Set RESUME_FROM = \'\' to train from scratch.'
        )
# Reproducibility: snapshot the full CONFIG + git SHA + start time to
# {TRAIN_OUTPUT_DIR}/config.json so months from now we know what produced
# this adapter. Done BEFORE training kicks off so even partial runs leave
# a config trace on Drive.
import json as _json_cfg, subprocess as _sp_cfg, time as _time_cfg, os as _os_cfg
_os_cfg.makedirs(TRAIN_OUTPUT_DIR, exist_ok=True)
_cfg_snapshot = {
    'BRANCH': BRANCH, 'HUB_USER': HUB_USER, 'RUN_NAME': RUN_NAME, 'BGE_MODEL': BGE_MODEL,
    'HUB_REPO': HUB_REPO, 'HUB_REPO_MERGED': HUB_REPO_MERGED,
    'TRIPLES_JSONL': TRIPLES_JSONL, 'TRAIN_OUTPUT_DIR': TRAIN_OUTPUT_DIR,
    'SKIP_MINING': SKIP_MINING, 'RESUME_FROM': RESUME_FROM, 'SEED': SEED,
    'MAX_INPUT_ROWS': MAX_INPUT_ROWS, 'DEV_EVAL_ROWS': DEV_EVAL_ROWS,
    'PERCPOS_THRESHOLD': PERCPOS_THRESHOLD, 'POOL_SIZE': POOL_SIZE,
    'N_NEGATIVES': N_NEGATIVES, 'MINING_BATCH_SIZE': MINING_BATCH_SIZE,
    'EPOCHS': EPOCHS, 'PER_DEVICE_BATCH_SIZE': PER_DEVICE_BATCH_SIZE,
    'GRAD_ACCUM_STEPS': GRAD_ACCUM_STEPS, 'LR': LR, 'TEMPERATURE': TEMPERATURE,
    'LORA_RANK': LORA_RANK, 'LORA_ALPHA': LORA_ALPHA,
    'LR_SCHEDULE': LR_SCHEDULE,
    'QUERY_MAX_LEN': QUERY_MAX_LEN, 'PASSAGE_MAX_LEN': PASSAGE_MAX_LEN,
    'SPLIT_KEY': SPLIT_KEY, 'VAL_FRACTION': VAL_FRACTION,
    'LOGGING_STEPS': LOGGING_STEPS, 'VAL_EVERY_N_STEPS': VAL_EVERY_N_STEPS,
    'VAL_FULL_CATALOG_EVERY_N': VAL_FULL_CATALOG_EVERY_N,
    'CHECKPOINT_EVERY_N_EPOCHS': CHECKPOINT_EVERY_N_EPOCHS,
    'GRADIENT_CHECKPOINTING': GRADIENT_CHECKPOINTING,
    # Multi-modal upgrade (fresh-model): all toggleable, default-off so the
    # text-only baseline reproduces bit-exactly when USE_MULTIMODAL=False.
    'USE_MULTIMODAL': USE_MULTIMODAL,
    'MULTIMODAL_ARTIFACTS': MULTIMODAL_ARTIFACTS if USE_MULTIMODAL else None,
    'USE_DISTILLATION': USE_DISTILLATION if USE_MULTIMODAL else False,
    'DISTILL_WEIGHT': DISTILL_WEIGHT,
    'USE_MULTIPOSITIVE': USE_MULTIPOSITIVE if USE_MULTIMODAL else False,
    'MULTIPOSITIVE_THRESHOLD': MULTIPOSITIVE_THRESHOLD,
    'MODALITY_DROPOUT': MODALITY_DROPOUT,
    'TEACHER_MODEL': TEACHER_MODEL if USE_MULTIMODAL else None,
    'TEACHER_SCORES_PATH': TEACHER_SCORES_PATH if USE_MULTIMODAL else None,
    'commit_sha': _sp_cfg.check_output(['git','-C','/content/recsys2026','rev-parse','HEAD']).decode().strip(),
    'started_at': _time_cfg.strftime('%Y-%m-%d %H:%M:%S %Z'),
}
with open(f'{TRAIN_OUTPUT_DIR}/config.json', 'w') as _f_cfg:
    _json_cfg.dump(_cfg_snapshot, _f_cfg, indent=2)
print(f'[train] CONFIG snapshot -> {TRAIN_OUTPUT_DIR}/config.json (commit '
      f'{_cfg_snapshot["commit_sha"][:8]})')
_grad_ckpt_flag = '--gradient-checkpointing' if GRADIENT_CHECKPOINTING else '--no-gradient-checkpointing'
_ckpt_flag = f'--checkpoint-every-n-epochs {CHECKPOINT_EVERY_N_EPOCHS}'
# Multi-modal flags (Phase 2d). Empty string when USE_MULTIMODAL=False so the
# shell command degrades to the text-only baseline byte-for-byte.
if USE_MULTIMODAL:
    _mm_train_flags = (
        f'--use-multimodal --multimodal-artifacts {MULTIMODAL_ARTIFACTS} '
        f'--modality-dropout {MODALITY_DROPOUT}'
    )
    if USE_DISTILLATION:
        _mm_train_flags += f' --use-distillation --distill-weight {DISTILL_WEIGHT}'
    if USE_MULTIPOSITIVE:
        _mm_train_flags += f' --use-multipositive --multipositive-threshold {MULTIPOSITIVE_THRESHOLD}'
    if USE_DISTILLATION or USE_MULTIPOSITIVE:
        _mm_train_flags += f' --teacher-scores-path {TEACHER_SCORES_PATH}'
    print(f'[train] MULTI-MODAL ON: {_mm_train_flags}')
else:
    _mm_train_flags = ''
    print('[train] text-only baseline (USE_MULTIMODAL=False)')
!cd /content/recsys2026 && python -u scripts/train_bi_encoder.py \
    --triples {TRIPLES_JSONL} \
    --base-model {BGE_MODEL} \
    --output-dir {TRAIN_OUTPUT_DIR} \
    --hub-repo {HUB_REPO} \
    --results-dir {DRIVE_RESULTS} \
    --epochs {EPOCHS} \
    --per-device-batch-size {PER_DEVICE_BATCH_SIZE} \
    --gradient-accumulation-steps {GRAD_ACCUM_STEPS} \
    --lr {LR} \
    --lr-schedule {LR_SCHEDULE} \
    --seed {SEED} \
    --temperature {TEMPERATURE} \
    --lora-rank {LORA_RANK} \
    --lora-alpha {LORA_ALPHA} \
    --split-key {SPLIT_KEY} \
    --val-fraction {VAL_FRACTION} \
    --query-max-len {QUERY_MAX_LEN} \
    --passage-max-len {PASSAGE_MAX_LEN} \
    --logging-steps {LOGGING_STEPS} \
    --val-every-n-steps {VAL_EVERY_N_STEPS} \
    --val-full-catalog-every-n-steps {VAL_FULL_CATALOG_EVERY_N} \
    {_ckpt_flag} \
    {_resume_flag} \
    {_grad_ckpt_flag} \
    {_mm_train_flags} \
    --merge --cleanup-after-push \
    2>&1 | tee {DRIVE_LOG_TRAIN}


In [ ]:
# 5) TensorBoard launcher — open in PARALLEL with cell 4.
# Compare for the leak check:
#   train/ndcg_inbatch  ← per-batch on training rows
#   val/ndcg            ← per-batch on held-out USER-DISJOINT val users
# Aligned curves (gap < ~0.05 sustained) → no leak.
# Also watch val/full_catalog_ndcg_at_20 — should climb above ~0.05.
%load_ext tensorboard
%tensorboard --logdir {TRAIN_OUTPUT_DIR}/runs --port={TENSORBOARD_PORT}


In [ ]:
# 6 multi-modal: split paths by USE_MULTIMODAL.
# - text-only (legacy): existing SentenceTransformer-based catalog re-embed.
# - multi-modal: calls scripts/embed_catalog_multimodal.py which uses
#   MultiModalBiEncoder.forward_track over CLAP + CF + tag + release tokens.
# Both paths write to CATALOG_OUT_DIR/track_embeddings.pkl so cell 7 finds it.
if USE_MULTIMODAL:
    import os as _os6
    _os6.makedirs(CATALOG_OUT_DIR, exist_ok=True)
    # The trained multi-modal model lives at TRAIN_OUTPUT_DIR/merged (after
    # cell 4's --merge --cleanup-after-push). If --cleanup-after-push removed
    # it, the upload still hit HUB_REPO_MERGED — load from there as fallback.
    _mm_model_dir = f'{TRAIN_OUTPUT_DIR}/merged'
    if not _os6.path.isdir(_mm_model_dir):
        _mm_model_dir = HUB_REPO_MERGED  # HF Hub fallback
    print(f'[catalog re-embed MM] loading from {_mm_model_dir}')
    !cd /content/recsys2026 && python -u scripts/embed_catalog_multimodal.py \
        --model-dir {_mm_model_dir} \
        --backbone {BGE_MODEL} \
        --multimodal-artifacts {MULTIMODAL_ARTIFACTS} \
        --catalog-out-dir {CATALOG_OUT_DIR} \
        --passage-max-len {PASSAGE_MAX_LEN} \
        --batch-size 64
else:
    # 6) Re-embed the ~47K-track catalog with the fine-tuned (merged) model.
    #
    # PATCH 4 (2026-05-22): catalog text now uses the SAME `id_to_metadata` format
    # the training builder writes for pos/neg AND the format `chat_history_parser`
    # uses to expand music-turn references in [HISTORY] blocks at inference. This
    # closes the multi-way format asymmetry surfaced by the BlindA nDCG diagnosis.
    # Old catalog vectors (built with format_track_text, 5 fields) are stale; this
    # cell overwrites them with the aligned-format version.
    import os, pickle, numpy as np, sys
    from datasets import load_dataset
    from sentence_transformers import SentenceTransformer
    sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
    sys.path.insert(0, '/content/recsys2026/scripts')
    from build_bi_encoder_training_data import _format_history_music_turn

    os.makedirs(CATALOG_OUT_DIR, exist_ok=True)
    model = SentenceTransformer(HUB_REPO_MERGED, device='cuda')
    model.max_seq_length = PASSAGE_MAX_LEN
    model.tokenizer.truncation_side = 'right'
    print(f'[catalog re-embed] max_seq_length={model.max_seq_length} truncation_side={model.tokenizer.truncation_side}')

    tm = load_dataset('talkpl-ai/TalkPlayData-Challenge-Track-Metadata', split='all_tracks')
    metadata_dict = {row['track_id']: dict(row) for row in tm}
    corpus_types = ['track_name', 'artist_name', 'album_name']  # MUST match production config 021 + training builder
    track_ids = [row['track_id'] for row in tm]
    texts = [_format_history_music_turn(tid, metadata_dict, corpus_types) for tid in track_ids]
    print(f'[catalog re-embed] format aligned with id_to_metadata; sample: {texts[0][:200]}')
    embs = model.encode(texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
    embs = np.asarray(embs, dtype=np.float32)
    out_path = os.path.join(CATALOG_OUT_DIR, 'track_embeddings.pkl')
    with open(out_path, 'wb') as f:
        pickle.dump({'track_ids': track_ids, 'track_mat': embs}, f)
    print(f'wrote {len(track_ids)} embeddings → {out_path}')


In [ ]:
# 7) Dev eval — uses the PRODUCTION code path (DENSE_LOCAL + build_retrieval_query
# + chat_history_parser-equivalent). Same path BlindA inference goes through, so
# the dev nDCG is apples-to-apples with the leaderboard's nDCG axis (modulo the
# rerank + responder stages, which apply equally to both).
#
# PATCH 2 (2026-05-22) + bugfix audit (2026-05-22):
# - Replaces the prior manual SentenceTransformer + cosine replica that had
#   subtle divergences from production.
# - FIX A: DENSE_LOCAL kwarg is `split_types`, not `track_split_types`.
# - FIX B: iterate USER turns and predict the NEXT music turn (mirrors
#   production's target_turn_number = user turn). Builds chat_history from
#   turns BEFORE the user turn — does NOT include the user turn itself —
#   so build_retrieval_query's last-user scan finds exactly one user_query
#   in session_memory (no duplicate).
import sys, math, os, json, time
import numpy as np
from datasets import load_dataset
sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
sys.path.insert(0, '/content/recsys2026/scripts')
from mcrs.crs_baseline import build_retrieval_query
from mcrs.retrieval_modules.dense_local import DENSE_LOCAL
from mcrs.db_item.music_catalog import MusicCatalogDB

# Preflight: HF train↔test session-disjointness contract.
train_sess = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='train')
test_sess  = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='test')
train_sids = {s.get('session_id') for s in train_sess if s.get('session_id')}
test_sids  = {s.get('session_id') for s in test_sess  if s.get('session_id')}
sess_overlap = train_sids & test_sids
if sess_overlap:
    raise AssertionError(f'HF train/test share {len(sess_overlap)} session_ids — leak. Refusing to score.')
print(f'[preflight] session-disjointness OK (train={len(train_sids):,}, test={len(test_sids):,})')

# Build MusicCatalogDB exactly as production does.
item_db = MusicCatalogDB(
    dataset_name='talkpl-ai/TalkPlayData-Challenge-Track-Metadata',
    split_types=['all_tracks'],
    corpus_types=['track_name', 'artist_name', 'album_name'],
)
print(f'[dev eval] item_db loaded: {len(item_db.metadata_dict):,} tracks')

# 7 multi-modal: branch on USE_MULTIMODAL.
# - text-only (legacy): DENSE_LOCAL with sentence-transformers query encoding.
# - multi-modal: DENSE_MULTIMODAL_LOCAL with user_id-aware query encoding
#   (cold users → train-set mean fallback).
if USE_MULTIMODAL:
    from mcrs.retrieval_modules.dense_multimodal_local import DENSE_MULTIMODAL_LOCAL
    # IMPORTANT: pass HUB_REPO_MERGED (NOT the local merged dir path) as model_dir.
    # DENSE_MULTIMODAL_LOCAL uses model_dir for BOTH model load AND catalog cache
    # key construction (_track_cache_path: {cache}/dense_local/{model_dir.replace
    # ("/","_")}/{embed_label}/track_embeddings.pkl). Cell 6 writes the catalog
    # to a path keyed on HUB_REPO_MERGED via CATALOG_OUT_DIR/SAFE_MODEL. Passing
    # a local absolute path here would mangle to a different safe_model and the
    # retriever wouldn't find the catalog. MultiModalBiEncoder.from_pretrained
    # auto-fetches from Hub if the path isn't local, so load works either way.
    retriever = DENSE_MULTIMODAL_LOCAL(
        dataset_name='talkpl-ai/TalkPlayData-Challenge-Track-Metadata',
        split_types=['all_tracks'],
        corpus_types=['track_name', 'artist_name', 'album_name'],
        cache_dir='/content/recsys2026/experiments/cache',
        model_dir=HUB_REPO_MERGED,
        embed_label=EMBED_LABEL,
        backbone_override=BGE_MODEL,
        multimodal_artifacts=MULTIMODAL_ARTIFACTS,
        query_max_len=QUERY_MAX_LEN,
    )
    print(f'[dev eval] DENSE_MULTIMODAL_LOCAL ready: model_dir={HUB_REPO_MERGED}, embed_label={EMBED_LABEL}')
else:
    retriever = DENSE_LOCAL(
        dataset_name='talkpl-ai/TalkPlayData-Challenge-Track-Metadata',
        split_types=['all_tracks'],
        corpus_types=['track_name', 'artist_name', 'album_name'],
        cache_dir='/content/recsys2026/experiments/cache',
        model_name=HUB_REPO_MERGED,
        embed_label=EMBED_LABEL,
    )
    print(f'[dev eval] DENSE_LOCAL ready: encoder={HUB_REPO_MERGED}, embed_label={EMBED_LABEL}')

def _build_history_for_user_at(convs, user_pos):
    """Build chat_history exactly as production's chat_history_parser does:
    all turns BEFORE the target user turn (user_pos), with music turns
    expanded via item_db.id_to_metadata. Does NOT include the user turn
    itself — that goes into session_memory separately as the appended
    {role:user, content:user_query} entry. This mirrors batch_chat which
    receives chat_history (no user_query) then appends user_query."""
    chat_history = []
    for prev in convs[:user_pos]:
        role = prev.get('role')
        content = prev.get('content') or ''
        if role == 'user':
            chat_history.append({'role': 'user', 'content': content})
        elif role == 'music':
            expanded = item_db.id_to_metadata(content) if content in item_db.metadata_dict else content
            chat_history.append({'role': 'assistant', 'content': expanded})
        elif role == 'assistant':
            chat_history.append({'role': 'assistant', 'content': content})
    return chat_history

# Walk test sessions. For each USER turn whose NEXT turn is a MUSIC turn,
# emit one (query, gold) pair. This matches the (user_query, target track)
# semantics that BlindA's target_turn_number marks.
queries_text, gold_tids = [], []
for sess in test_sess:
    convs = sess.get('conversations', [])
    for user_pos, turn in enumerate(convs):
        if turn.get('role') != 'user':
            continue
        # Need a music turn immediately after to have a gold.
        if user_pos + 1 >= len(convs) or convs[user_pos + 1].get('role') != 'music':
            continue
        user_query = turn.get('content') or ''
        gold = convs[user_pos + 1].get('content')
        if not user_query or not gold:
            continue
        history = _build_history_for_user_at(convs, user_pos)
        # session_memory = chat_history + appended user_query. This is what
        # batch_chat does at production (crs_baseline.py:445-449).
        session_memory = history + [{'role': 'user', 'content': user_query}]
        cg = sess.get('conversation_goal') or {}
        goal_text = (cg.get('listener_goal') or '').strip() or None
        qtext = build_retrieval_query(
            session_memory,
            mode='bge_m3_structured',
            goal_text=goal_text,
            user_profile=sess.get('user_profile'),
        )
        queries_text.append(qtext)
        gold_tids.append(gold)
        if len(queries_text) >= DEV_EVAL_ROWS:
            break
    if len(queries_text) >= DEV_EVAL_ROWS:
        break

print(f'[dev eval] {len(queries_text)} queries built via production code path')
print(f'[dev eval] sample query[:400]:\n  {queries_text[0][:400]}')

# Score via DENSE_LOCAL.
# Multi-modal needs user_ids parallel to queries; legacy DENSE_LOCAL ignores them.
_session_user_ids = []
if USE_MULTIMODAL:
    # Recompute the user_id list by re-walking test_sess in the same order
    # used to build queries_text above. We tracked nothing per-query so far —
    # do the cheap rewalk here. (Adds ~5 sec for 8000 queries.)
    _budget = len(queries_text)
    for sess in test_sess:
        if len(_session_user_ids) >= _budget:
            break
        convs = sess.get('conversations', [])
        sess_uid = sess.get('user_id')
        for user_pos, turn in enumerate(convs):
            if len(_session_user_ids) >= _budget:
                break
            if turn.get('role') != 'user':
                continue
            if user_pos + 1 >= len(convs) or convs[user_pos + 1].get('role') != 'music':
                continue
            user_query = turn.get('content') or ''
            gold = convs[user_pos + 1].get('content')
            if not user_query or not gold:
                continue
            _session_user_ids.append(str(sess_uid) if sess_uid is not None else None)
    assert len(_session_user_ids) == len(queries_text), (
        f'user_id rewalk drifted: got {len(_session_user_ids)} vs {len(queries_text)} queries'
    )
    print(f'[dev eval MM] user_ids: {sum(1 for u in _session_user_ids if u)} warm '
          f'/ {sum(1 for u in _session_user_ids if u is None)} cold')
    top20 = retriever.batch_text_to_item_retrieval(queries_text, topk=20, user_ids=_session_user_ids)
else:
    top20 = retriever.batch_text_to_item_retrieval(queries_text, topk=20)
# I4: persist any cache entries built up during the eval pass.
if hasattr(retriever, 'flush_query_cache'):
    retriever.flush_query_cache()
ndcgs = []
for gold, ranked in zip(gold_tids, top20):
    if gold in ranked:
        rank = ranked.index(gold) + 1
        ndcgs.append(1.0 / math.log2(rank + 1))
    else:
        ndcgs.append(0.0)
mean_ndcg = float(sum(ndcgs) / max(1, len(ndcgs)))
_dev_retriever_label = 'DENSE_MULTIMODAL_LOCAL' if USE_MULTIMODAL else 'DENSE_LOCAL'
print(f'\n[dev eval] standalone {_dev_retriever_label} nDCG@20 = {mean_ndcg:.4f} (n={len(ndcgs)})')
print(f'[dev eval] this number is APPLES-TO-APPLES with the production code path.')

# Persist.
_dev_eval = {
    'run_name': RUN_NAME, 'hub_repo': HUB_REPO_MERGED,
    'epochs': EPOCHS, 'seed': SEED,
    'mining_strategy': MINING_STRATEGY,
    'dev_eval_rows': len(ndcgs),
    'standalone_ndcg_at_20': mean_ndcg,
    'eval_path': f'{_dev_retriever_label} (production code path)',
    'completed_at': time.strftime('%Y-%m-%d %H:%M:%S %Z'),
}
os.makedirs(TRAIN_OUTPUT_DIR, exist_ok=True)
with open(f'{TRAIN_OUTPUT_DIR}/dev_eval.json', 'w') as f:
    json.dump(_dev_eval, f, indent=2)
print(f'[dev eval] result → {TRAIN_OUTPUT_DIR}/dev_eval.json')


In [ ]:
# 7b) Multi-modal parity gates (Phase 5). Run BEFORE trusting any nDCG
# from cell 7 when USE_MULTIMODAL=True. Quick (~30 sec) diagnostics that
# catch the high-impact failure modes:
#   1. CF leakage (zero user_cf and check if score drops meaningfully).
#   2. CLAP source consistency (single artifact dir feeds both train + inference).
#   3. <user_cf> token substitution actually changes query embedding.
#   4. Cold-user fraction in dev matches Blind-A within 5pp.
#   5. Training-builder query text vs production build_retrieval_query format.
if not USE_MULTIMODAL:
    print('[parity gates] skipping — USE_MULTIMODAL=False')
else:
    import hashlib as _hash_pg, math as _math_pg, numpy as _np_pg, torch as _torch_pg
    from datasets import load_dataset as _ld_pg
    from mcrs.retrieval_modules.bge_m3_format import format_query_text as _fqt

    def _quick_ndcg(top20s, golds):
        n = 0.0
        for t, g in zip(top20s, golds):
            if g in t:
                n += 1.0 / _math_pg.log2(t.index(g) + 2)
        return n / max(1, len(golds))

    sample_n = min(200, len(queries_text))
    sq, sg, su = queries_text[:sample_n], gold_tids[:sample_n], _session_user_ids[:sample_n]

    print('=== Gate 1: CF leakage (zero user_cf vs warm) ===')
    top_warm = retriever.batch_text_to_item_retrieval(sq, topk=20, user_ids=su)
    _orig_get_uc = retriever._get_user_cf
    _zero_uc = _np_pg.zeros_like(retriever._user_cf['mean'])
    retriever._get_user_cf = lambda uid: _zero_uc
    retriever._query_cache.clear()
    top_zero = retriever.batch_text_to_item_retrieval(sq, topk=20, user_ids=su)
    retriever._get_user_cf = _orig_get_uc
    retriever._query_cache.clear()
    n_warm, n_zero = _quick_ndcg(top_warm, sg), _quick_ndcg(top_zero, sg)
    print(f'  warm user_cf nDCG@20 = {n_warm:.4f}')
    print(f'  zero user_cf nDCG@20 = {n_zero:.4f}')
    print(f'  delta = {abs(n_warm - n_zero):.4f}')
    if abs(n_warm - n_zero) < 0.005:
        print('  ⚠️  CF signal contributes <0.005 nDCG on this sample. Could be:')
        print('     (a) CF leakage (user embeddings include test-time signal)')
        print('     (b) Cold-user-dominated sample (most uids → mean fallback)')
        print('     Investigate before trusting cell 7 results.')
    else:
        print(f'  ✅ CF signal alive ({abs(n_warm - n_zero):.4f} nDCG contribution)')

    print()
    print('=== Gate 2: CLAP source consistency ===')
    _clap_path = f'{MULTIMODAL_ARTIFACTS}/track_clap.npy'
    with open(_clap_path, 'rb') as _f:
        _clap_hash = _hash_pg.sha256(_f.read()).hexdigest()[:16]
    print(f'  artifact: {_clap_path}')
    print(f'  sha256:   {_clap_hash}')
    print(f'  Training (TripleJsonlDataset.MultiModalArtifacts) reads from this path.')
    print(f'  Inference (embed_catalog_multimodal.py via cell 6) reads from this path.')
    print('  ✅ Same artifact dir → train/inference parity by construction')

    print()
    print('=== Gate 3: <user_cf> token substitution alive ===')
    _model_pg, _tok_pg = retriever._get_model_and_tokenizer()
    _device_pg = next(_model_pg.parameters()).device
    _cf_dim = retriever._user_cf['dim']
    _uc_a = _np_pg.zeros(_cf_dim, dtype=_np_pg.float32); _uc_a[0] = 1.0
    _uc_b = _np_pg.zeros(_cf_dim, dtype=_np_pg.float32); _uc_b[min(1, _cf_dim-1)] = 1.0
    _enc_pg = _tok_pg(['parity test query'], max_length=32, padding=True,
                       truncation=True, return_tensors='pt')
    _enc_pg = {k: v.to(_device_pg) for k, v in _enc_pg.items()}
    with _torch_pg.no_grad():
        _e_a = _model_pg.forward_query(
            input_ids=_enc_pg['input_ids'], attention_mask=_enc_pg['attention_mask'],
            user_cf=_torch_pg.from_numpy(_uc_a).unsqueeze(0).to(_device_pg))
        _e_b = _model_pg.forward_query(
            input_ids=_enc_pg['input_ids'], attention_mask=_enc_pg['attention_mask'],
            user_cf=_torch_pg.from_numpy(_uc_b).unsqueeze(0).to(_device_pg))
    _diff = (_e_a - _e_b).norm().item()
    print(f'  embedding L2 diff (different user_cf): {_diff:.6f}')
    if _diff < 0.001:
        print('  ⚠️  <user_cf> token has NO effect — substitution path broken!')
    else:
        print(f'  ✅ user_cf shapes the query embedding ({_diff:.4f} diff)')

    print()
    print('=== Gate 4: Cold-user fraction (HF test vs Blind-A) ===')
    _warm_dev = sum(1 for u in _session_user_ids if u and u in retriever._user_cf['uid_to_idx'])
    _dev_cold_pct = 100.0 * (1 - _warm_dev / max(1, len(_session_user_ids)))
    print(f'  HF test (cell 7 eval set) cold-user rate: {_dev_cold_pct:.1f}%')
    try:
        _blind = _ld_pg('talkpl-ai/TalkPlayData-Challenge-Blind-A', split='test')
        _blind_uids = [str(s.get('user_id')) if s.get('user_id') is not None else None for s in _blind]
        _warm_blind = sum(1 for u in _blind_uids if u and u in retriever._user_cf['uid_to_idx'])
        _blind_cold_pct = 100.0 * (1 - _warm_blind / max(1, len(_blind_uids)))
        print(f'  Blind-A cold-user rate:                    {_blind_cold_pct:.1f}%')
        _pp = abs(_dev_cold_pct - _blind_cold_pct)
        if _pp > 5.0:
            print(f'  ⚠️  >5pp distribution gap ({_pp:.1f}pp). Cold-user calibration off.')
        else:
            print(f'  ✅ Within {_pp:.1f}pp — calibrated for inference distribution')
    except Exception as _e:
        print(f'  [skip] Blind-A load failed: {type(_e).__name__}: {_e}')

    print()
    print('=== Gate 5: train-builder vs production-builder query format ===')
    _n_diffs = 0
    for _sess in test_sess.select(range(min(5, len(test_sess)))):
        _convs = _sess.get('conversations', [])
        _user_pos = None
        for _i, _t in enumerate(_convs):
            if _t.get('role') == 'user' and _i + 1 < len(_convs) and _convs[_i+1].get('role') == 'music':
                _user_pos = _i
                break
        if _user_pos is None:
            continue
        _uq = _convs[_user_pos].get('content', '')
        if not _uq:
            continue
        # Training-time format: same construction the data builder uses.
        # Note: builder passes the FULL chat_history (already expanded); the
        # production path expands music turns via _build_history_for_user_at.
        _hist_prod = _build_history_for_user_at(_convs, _user_pos)
        _cg = _sess.get('conversation_goal') or {}
        _goal = (_cg.get('listener_goal') or '').strip() or None
        _train_q = _fqt(chat_history=_hist_prod,
                         current_user_query=_uq,
                         user_profile=_sess.get('user_profile'),
                         conversation_goal=_sess.get('conversation_goal'),
                         mode='bge_m3_structured')
        _prod_q = build_retrieval_query(
            _hist_prod + [{'role':'user', 'content':_uq}],
            mode='bge_m3_structured', goal_text=_goal,
            user_profile=_sess.get('user_profile'),
        )
        if _train_q == _prod_q:
            continue
        _n_diffs += 1
        for _idx, (_a, _b) in enumerate(zip(_train_q, _prod_q)):
            if _a != _b:
                print(f'  ⚠️  DIFF: q[:40]={_uq[:40]!r}')
                print(f'      train: ...{_train_q[max(0,_idx-15):_idx+30]!r}...')
                print(f'      prod:  ...{_prod_q[max(0,_idx-15):_idx+30]!r}...')
                break
        else:
            print(f'  ⚠️  LENGTH DIFF: train={len(_train_q)} prod={len(_prod_q)}')
    if _n_diffs == 0:
        print('  ✅ all 5 sampled queries: byte-identical')

    print()
    print('=== PARITY GATES COMPLETE ===')


In [ ]:
# 7c) Modality ablation (Phase 5). Verify each modality contributes
# ≥0.005 dev nDCG. Drops any modality that doesn't pull weight before
# Stage B training.
#
# user_cf ablation is in-memory + fast (~10 sec). Track-side ablations
# (audio / cf / tag / release) each require a fresh catalog re-encode
# via scripts/embed_catalog_multimodal.py with --ablate-modality (added
# in Phase 5a). They take ~3-5 min each on Blackwell, so they're guarded
# behind RUN_TRACK_ABLATIONS (set False here; opt in by editing).
if not USE_MULTIMODAL:
    print('[ablation] skipping — USE_MULTIMODAL=False')
else:
    import math as _m_ab, numpy as _np_ab, os as _os_ab
    RUN_TRACK_ABLATIONS = False   # set True to do the 4 expensive re-encodes

    def _ablation_ndcg(top20s, golds):
        n = 0.0
        for t, g in zip(top20s, golds):
            if g in t:
                n += 1.0 / _m_ab.log2(t.index(g) + 2)
        return n / max(1, len(golds))

    # Baseline from cell 7's run (mean_ndcg variable).
    print(f'baseline (from cell 7): nDCG@20 = {mean_ndcg:.4f} (n={len(gold_tids)})')
    print()

    # ----- user_cf ablation (in-memory monkey-patch) -----
    print('--- Ablating user_cf (query side, no re-encode needed) ---')
    _orig_get_uc = retriever._get_user_cf
    _zero_uc = _np_ab.zeros_like(retriever._user_cf['mean'])
    retriever._get_user_cf = lambda uid: _zero_uc
    retriever._query_cache.clear()
    _top_no_uc = retriever.batch_text_to_item_retrieval(
        queries_text, topk=20, user_ids=_session_user_ids,
    )
    retriever._get_user_cf = _orig_get_uc
    retriever._query_cache.clear()
    _ndcg_no_uc = _ablation_ndcg(_top_no_uc, gold_tids)
    _delta_uc = mean_ndcg - _ndcg_no_uc
    print(f'  user_cf → 0 :  nDCG@20 = {_ndcg_no_uc:.4f}  (Δ = {_delta_uc:+.4f})')
    _verdict_uc = '✅ keep' if _delta_uc >= 0.005 else '⚠️  drop (lift <0.005)'
    print(f'  verdict: {_verdict_uc}')
    print()

    # ----- track-side ablations (re-encode catalog per modality) -----
    _track_results = {}
    if RUN_TRACK_ABLATIONS:
        print('--- Track-side ablations: each re-encodes the catalog (~3-5 min) ---')
        from mcrs.retrieval_modules.dense_multimodal_local import DENSE_MULTIMODAL_LOCAL
        _mm_dir = f'{TRAIN_OUTPUT_DIR}/merged'
        if not _os_ab.path.isdir(_mm_dir):
            _mm_dir = HUB_REPO_MERGED
        for _mod in ('audio', 'cf', 'tag', 'release'):
            print(f'  ablating {_mod}:')
            # C2 fix: write the ablated catalog to EXACTLY the path
            # DENSE_MULTIMODAL_LOCAL._track_cache_path() will construct from
            # (model_dir=HUB_REPO_MERGED, embed_label=_ablate_label). Keeping
            # model_dir the same as baseline correctly shares the query cache
            # (query encoding is unchanged by track-side ablation — only the
            # catalog encoder sees the masked modality).
            _ablate_label = f'{EMBED_LABEL}__ablate_{_mod}'
            _ablate_dir = f'{CACHE_ROOT}/{SAFE_MODEL}/{_ablate_label}'
            _os_ab.makedirs(_ablate_dir, exist_ok=True)
            !cd /content/recsys2026 && python -u scripts/embed_catalog_multimodal.py \
                --model-dir {_mm_dir} \
                --backbone {BGE_MODEL} \
                --multimodal-artifacts {MULTIMODAL_ARTIFACTS} \
                --catalog-out-dir {_ablate_dir} \
                --ablate-modality {_mod} \
                --passage-max-len {PASSAGE_MAX_LEN} \
                --batch-size 64 2>&1 | tail -5
            _ab_retriever = DENSE_MULTIMODAL_LOCAL(
                dataset_name='talkpl-ai/TalkPlayData-Challenge-Track-Metadata',
                split_types=['all_tracks'],
                corpus_types=['track_name', 'artist_name', 'album_name'],
                cache_dir='/content/recsys2026/experiments/cache',
                model_dir=HUB_REPO_MERGED,
                embed_label=_ablate_label,
                backbone_override=BGE_MODEL,
                multimodal_artifacts=MULTIMODAL_ARTIFACTS,
                query_max_len=QUERY_MAX_LEN,
            )
            _top_ab = _ab_retriever.batch_text_to_item_retrieval(
                queries_text, topk=20, user_ids=_session_user_ids,
            )
            _ndcg_ab = _ablation_ndcg(_top_ab, gold_tids)
            _delta_ab = mean_ndcg - _ndcg_ab
            _verdict_ab = '✅ keep' if _delta_ab >= 0.005 else '⚠️  drop'
            print(f'    {_mod} → 0 :  nDCG@20 = {_ndcg_ab:.4f}  (Δ = {_delta_ab:+.4f}) {_verdict_ab}')
            _track_results[_mod] = (_ndcg_ab, _delta_ab)
    else:
        print('--- Track-side ablations: SKIPPED (RUN_TRACK_ABLATIONS=False) ---')
        print('    Set RUN_TRACK_ABLATIONS=True above to re-encode the catalog with')
        print('    one modality zeroed at a time. ~3-5 min/modality on Blackwell.')

    print()
    print('=== ABLATION SUMMARY ===')
    print(f'  baseline:           {mean_ndcg:.4f}')
    print(f'  ablate user_cf :    {_ndcg_no_uc:.4f}  (Δ={_delta_uc:+.4f}) {_verdict_uc}')
    for _mod, (_n, _d) in _track_results.items():
        _v = '✅' if _d >= 0.005 else '⚠️'
        print(f'  ablate {_mod:8}:    {_n:.4f}  (Δ={_d:+.4f}) {_v}')
    if not _track_results:
        print('  ablate audio   :    [run with RUN_TRACK_ABLATIONS=True]')
        print('  ablate cf      :    [run with RUN_TRACK_ABLATIONS=True]')
        print('  ablate tag     :    [run with RUN_TRACK_ABLATIONS=True]')
        print('  ablate release :    [run with RUN_TRACK_ABLATIONS=True]')

    print()
    print('Gate to ship: dev nDCG@20 ≥ 0.16 + every modality contributes ≥0.005.')
    print('Modalities below threshold should be dropped from the deployed model')
    print('(fewer moving parts → faster inference, easier to debug).')
